In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


def calculate_dataset_statistics(
    dataset_path,
    image_size=(64, 64),
    batch_size=32,
    num_workers=0,
):
    """
    Bir ImageFolder dataset'i için RGB mean ve std hesaplar.

    Args:
        dataset_path (str): Dataset klasör yolu.
        image_size (tuple): Resize edilecek boyut.
        batch_size (int): DataLoader batch size.
        num_workers (int): DataLoader worker sayısı.

    Returns:
        mean (Tensor)
        std (Tensor)
    """

    # Sadece Resize ve ToTensor uygulanır.
    # Normalize uygulanmaz!
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor()
    ])

    dataset = datasets.ImageFolder(
        root=dataset_path,
        transform=transform
    )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_images = 0

    for images, _ in dataloader:

        # (B,C,H,W) -> (B,C,H*W)
        images = images.view(images.size(0), images.size(1), -1)

        mean += images.mean(dim=2).sum(dim=0)
        std += images.std(dim=2).sum(dim=0)

        total_images += images.size(0)

    mean /= total_images
    std /= total_images

    print("=" * 50)
    print(f"Dataset Path : {dataset_path}")
    print(f"Toplam Resim : {len(dataset)}")
    print(f"Sınıflar     : {dataset.classes}")
    print(f"Image Size   : {image_size}")
    print("=" * 50)

    print("\nMean:")
    print(mean)

    print("\nStd:")
    print(std)

    print("\nKopyala-Yapıştır Normalize:")
    print(f"""
transforms.Normalize(
    mean={mean.tolist()},
    std={std.tolist()}
)
""")

    return mean, std

Kullanımı

In [ ]:
mean, std = calculate_dataset_statistics(
    "data/dataAdi/train",
    image_size=(128,128)
)

In [ ]:
# kullanım için data yolunu ve image_size değerini değiştirmek yeterli olur